## Step 1 — ติดตั้ง Libraries

In [1]:
!pip install -q datasets
!pip install -q llama-index
!pip install -q llama-index-llms-huggingface
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-vector-stores-faiss
!pip install -q accelerate
!pip install -q bitsandbytes
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q diskcache
print("✅ ติดตั้งสำเร็จ")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.0/107.0 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.4/333.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.1

In [13]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 119.8 MB/s eta 0:00:00


## Step 2 — Import Libraries

In [2]:
import torch
import faiss
import diskcache as dc
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from llama_index.core import VectorStoreIndex, Document, Settings, StorageContext
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.faiss import FaissVectorStore
print("✅ Import สำเร็จ")


✅ Import สำเร็จ


## Step 3 — โหลด Dataset

In [3]:
raw = load_dataset('ChokF/thai_food')
df = raw['train'].to_pandas()

print('Columns:', df.columns.tolist())
print('จำนวนข้อมูล:', len(df))
df.head(3)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/64.0 [00:00<?, ?B/s]

thai_food_dataset.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1023 [00:00<?, ? examples/s]

Columns: ['ชื่อเมนู', 'วัตถุดิบ (Ingredients)', 'เครื่องปรุง (Seasonings)', 'วิธีทำ (Instructions)']
จำนวนข้อมูล: 1023


,ชื่อเมนู,วัตถุดิบ (Ingredients),เครื่องปรุง (Seasonings),วิธีทำ (Instructions)
0,ต้มยำกุ้งน้ำข้น,"กุ้งแม่น้ำ, เห็ดฟาง, ข่า, ตะไคร้, ใบมะกรูด, พร...","น้ำพริกเผา, นมข้นจืด, น้ำปลา, มะนาว, น้ำตาลทราย",1.ต้มน้ำกับสมุนไพรจนหอม 2.ใส่กุ้งและเห็ด 3.ผสม...
1,แกงเขียวหวานไก่,"เนื้ออกไก่, มะเขือเปาะ, มะเขือพวง, ใบโหระพา, พ...","พริกแกงเขียวหวาน, น้ำปลา, น้ำตาลปี๊บ",1.ผัดพริกแกงกับกะทิจนแตกมัน 2.ใส่ไก่ลงไปผัดจนส...
2,ผัดไทยกุ้งสด,"เส้นเล็ก, กุ้งสด, เต้าหู้เหลือง, ถั่วงอก, ใบกุ...","น้ำมะขามเปียก, น้ำตาลปี๊บ, น้ำปลา, พริกป่น",1.ผัดกุ้งให้สุกพักไว้ 2.ผัดเต้าหู้และเส้นกับน้...


## Step 4 — แปลง Dataset เป็น Documents

In [4]:
documents = []
for _, row in df.iterrows():
    parts = []
    for col in df.columns:
        if pd.notna(row[col]) and str(row[col]).strip():
            parts.append(f"{col}: {row[col]}")
    text = "\n".join(parts)
    documents.append(Document(text=text))

print(f'✅ สร้าง Documents: {len(documents)} รายการ')
print(documents[0].text[:300])


✅ สร้าง Documents: 1023 รายการ
ชื่อเมนู: ต้มยำกุ้งน้ำข้น
วัตถุดิบ (Ingredients): กุ้งแม่น้ำ, เห็ดฟาง, ข่า, ตะไคร้, ใบมะกรูด, พริกขี้หนูสวน
เครื่องปรุง (Seasonings): น้ำพริกเผา, นมข้นจืด, น้ำปลา, มะนาว, น้ำตาลทราย
วิธีทำ (Instructions): 1.ต้มน้ำกับสมุนไพรจนหอม 2.ใส่กุ้งและเห็ด 3.ผสมพริกเผากับนมข้นจืดเทลงไป 4.ปิดไฟปรุงรสด้วยมะนาวน้


## Step 5 — ตั้งค่า Embedding Model

In [5]:
# ใช้ BGE-M3 รองรับภาษาไทยได้ดีกว่า wangchanberta สำหรับงาน RAG
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3",
    embed_batch_size=32,
    max_length=512,
)
Settings.chunk_size    = 512
Settings.chunk_overlap = 40

print("✅ Embedding Model พร้อมใช้งาน")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Embedding Model พร้อมใช้งาน


## Step 6 — ตั้งค่า LLM (Chinda)

In [6]:
model_name = "iapp/chinda-qwen3-4b"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("⏳ กำลังโหลด Model (อาจใช้เวลาสักครู่)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)
print("✅ โหลด Model สำเร็จ")

system_prompt = (
    "คุณคือผู้ช่วยค้นหาและแนะนำวิธีทำอาหารไทย ที่มีข้อมูลใน documents\n"
    "ให้ตอบโดยมีการทวนหัวข้อเสมอ\n"
    "ตอบเป็นภาษาไทยเสมอ\n"
    "ตอบคำถามแค่ที่ถาม เช่น ถามวิธีทำตอบแค่วิธีทำ ถามวัตถุดิบตอบแค่วัตถุดิบ\n"
    "ขอคำตอบแค่3หัวข้อ คือ วัตถุดิบ เครื่องปรุงวิธีทำ\n"
    "ใช้ข้อมูล วัตถุดิบ เครื่องปรุง วิธีทำ แค่ใน documents เท่านั้า\n"
    "ให้ใช้เฉพาะข้อมูลที่ได้รับใน documents เท่านั้น\n"
    "ห้ามตอบจากความรู้ของตัวเองโดยเด็ดขาด\n"
    "ถ้าไม่มีข้อมูลใน documents ให้ตอบว่าไม่พบข้อมูล\n"

)

def messages_to_prompt(messages):
    """แปลง messages format ให้ตรงกับ Chinda/Qwen3 chat template"""
    full_messages = [{"role": "system", "content": system_prompt}] + list(messages)
    text = tokenizer.apply_chat_template(
        full_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False  # ปิด thinking mode เพื่อความเร็ว (เปิดได้ถ้าต้องการ)
    )
    return text

def completion_to_prompt(prompt):
    """สำหรับ query แบบ single string"""
    messages = [{"role": "user", "content": prompt}]
    return messages_to_prompt(messages)

llm = HuggingFaceLLM(
    model=model,
    tokenizer=tokenizer,
    context_window=4096,
    max_new_tokens=512,
    generate_kwargs={
        "temperature": 0.6,
        "top_p": 0.95,
        "top_k": 20,
        "do_sample": True,
    },
    messages_to_prompt=messages_to_prompt,
    completion_to_prompt=completion_to_prompt,
    is_chat_model=True,
)

Settings.llm = llm
print("✅ LLM พร้อมใช้งาน")


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

✅ โหลด Model สำเร็จ
✅ LLM พร้อมใช้งาน


## Step 7 — สร้าง FAISS Vector Index

In [7]:
# หา dimension จาก embedding จริง
test_embed  = Settings.embed_model.get_text_embedding("test")
dimension   = len(test_embed)
print(f"📐 Embedding dimension: {dimension}")

faiss_index     = faiss.IndexFlatL2(dimension)
vector_store    = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    show_progress=True
)
# เปลี่ยน chat_mode เป็น context_only
chat_engine = index.as_chat_engine(
    chat_mode="context",        # ← ใช้แค่ retrieved context, ไม่มีการสรุปประวัติ
    similarity_top_k=3,
    streaming=True,
    verbose=True,               # ← เปิดไว้ดูว่า retrieve มาจากไหน
)

#chat_engine = index.as_chat_engine(
    #chat_mode="condense_plus_context",
    #similarity_top_k=3,
    #streaming=True,
    #verbose=False
#)

print("✅ RAG System พร้อมใช้งาน")


📐 Embedding dimension: 1024


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/1027 [00:00<?, ?it/s]

✅ RAG System พร้อมใช้งาน


## Step 8 — ทดสอบ Chat

In [8]:
print("พิมพ์ 'ออก' หรือ 'exit' เพื่อหยุด")
print("=" * 50)

while True:
    user_message = input("🧑 ผู้ใช้: ").strip()

    if not user_message:
        continue

    if user_message.lower() in ["exit", "quit", "ออก", "หยุด"]:
        print("🤖 บอท: ลาก่อนครับ!")
        break

    print("🤖 บอท: ", end="", flush=True)
    response = chat_engine.stream_chat(user_message)
    for token in response.response_gen:
        print(token, end="", flush=True)
    print("\n" + "-" * 50)


พิมพ์ 'ออก' หรือ 'exit' เพื่อหยุด
🧑 ผู้ใช้: ออก
🤖 บอท: ลาก่อนครับ!


## Step 9 — วัด Accuracy (Precision@K)

## สร้าง เว็บ

In [23]:
index.storage_context.persist(persist_dir="./rag_storage")

# 2. บีบอัดเป็นไฟล์ zip เพื่อให้ดาวน์โหลดง่าย
!zip -r rag_storage.zip ./rag_storage

print("✅ บันทึกและสร้างไฟล์ rag_storage.zip สำเร็จ (ดาวน์โหลดได้ที่แถบไฟล์ด้านซ้าย)")

updating: rag_storage/ (stored 0%)
updating: rag_storage/docstore.json (deflated 84%)
updating: rag_storage/image__vector_store.json (deflated 19%)
updating: rag_storage/default__vector_store.json (deflated 7%)
updating: rag_storage/index_store.json (deflated 50%)
updating: rag_storage/graph_store.json (stored 0%)
✅ บันทึกและสร้างไฟล์ rag_storage.zip สำเร็จ (ดาวน์โหลดได้ที่แถบไฟล์ด้านซ้าย)
